In [1]:
import pandas as pd
import numpy as np

In [2]:
def assign_sessions(df, user_col='user_id', time_col='event_time',
                    session_timeout='30min'):
    df = df.copy()
    df[time_col] = pd.to_datetime(df[time_col])
    df = df.sort_values([user_col, time_col]).reset_index(drop=True)

    delta = df.groupby(user_col)[time_col].diff()

    new_session = (delta.isna()) | (delta > pd.Timedelta(session_timeout))
    df['is_new_session'] = new_session.astype(int)

    df['session_num'] = (df.groupby(user_col)['is_new_session']
                         .cumsum())

    df['session_id'] = (df[user_col].astype(str) + '_'
                        + df['session_num'].astype(str))
    return df

In [3]:
events = pd.read_json('data/events_lines.json', lines=True)
events['event_time'] = pd.to_datetime(events['event_time'])

events = assign_sessions(events)
print("Пример разметки сессий:")
print(events[['event_id', 'user_id', 'event_time', 'event_type',
              'session_id', 'session_num']].head(10))

sessions = (events.sort_values(['session_id', 'event_time'])
            .groupby('session_id')
            .agg(
                user_id=('user_id', 'first'),
                session_start=('event_time', 'min'),
                session_end=('event_time', 'max'),
                last_event_type=('event_type', 'last'),
                n_events=('event_id', 'count'),
                has_purchase=('event_type', lambda s: (s == 'purchase').any())
            )
            .reset_index())

sessions['is_purchase_session'] = sessions['last_event_type'] == 'purchase'

print("\n4.1) Завершилась ли сессия событием purchase:")
print(sessions.head(10).to_string(index=False))
print("\nВсего сессий:", len(sessions))
print("Из них завершились purchase:", sessions['is_purchase_session'].sum())

Пример разметки сессий:
   event_id  user_id          event_time   event_type session_id  session_num
0      2341     1001 2024-02-01 20:18:15        click     1001_1            1
1      3967     1001 2024-02-02 13:30:12    page_view     1001_2            2
2      3339     1001 2024-02-03 07:57:09       scroll     1001_3            3
3      2855     1001 2024-02-05 10:33:28     purchase     1001_4            4
4      3931     1001 2024-02-08 13:01:30    page_view     1001_5            5
5      2311     1001 2024-02-12 09:52:38  add_to_cart     1001_6            6
6      3702     1001 2024-02-15 10:34:34  add_to_cart     1001_7            7
7      2991     1001 2024-02-19 00:22:09     purchase     1001_8            8
8      3531     1001 2024-02-19 21:01:37     purchase     1001_9            9
9      3388     1001 2024-02-19 22:20:51        click    1001_10           10

4.1) Завершилась ли сессия событием purchase:
session_id  user_id       session_start         session_end last_event_

In [4]:
user_stats = (sessions.groupby('user_id')
              .agg(
                  total_sessions=('session_id', 'count'),
                  purchase_sessions=('is_purchase_session', 'sum')
              )
              .reset_index())

user_stats['purchase_session_share'] = (
    user_stats['purchase_sessions'] / user_stats['total_sessions']
)

user_stats = user_stats.sort_values('purchase_session_share',
                                    ascending=False).reset_index(drop=True)

print("4.2 Покупающие сессии :")
print(user_stats.to_string(index=False))

4.2 Покупающие сессии :
 user_id  total_sessions  purchase_sessions  purchase_session_share
    1058              23                 11                0.478261
    1001              28                 11                0.392857
    1091              31                 12                0.387097
    1003              21                  8                0.380952
    1004              21                  8                0.380952
    1071              17                  6                0.352941
    1005              21                  7                0.333333
    1050              15                  5                0.333333
    1057              19                  6                0.315789
    1036              19                  6                0.315789
    1066              26                  8                0.307692
    1046              23                  7                0.304348
    1078              17                  5                0.294118
    1002              24

In [5]:
top_users = (user_stats
             .sort_values(['purchase_session_share', 'purchase_sessions'],
                          ascending=False)
             .head(5))

print("Самые покупающие юзеры:")
print(top_users.to_string(index=False))

best = user_stats.iloc[0]
print(f"\nЛидер: user_id={best['user_id']}, "
      f"доля={best['purchase_session_share']:.2%} "
      f"({best['purchase_sessions']} из {best['total_sessions']} сессий)")

Самые покупающие юзеры:
 user_id  total_sessions  purchase_sessions  purchase_session_share
    1058              23                 11                0.478261
    1001              28                 11                0.392857
    1091              31                 12                0.387097
    1003              21                  8                0.380952
    1004              21                  8                0.380952

Лидер: user_id=1058.0, доля=47.83% (11.0 из 23.0 сессий)
